In [1]:
# pip install fastmcp
# pip install mcp

import os
from langchain_openai import ChatOpenAI   # openai接口
from langchain_ollama import ChatOllama   # ollama接口

from langgraph.checkpoint.memory import InMemorySaver # 记忆
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# MCP
from fastmcp import FastMCP
import subprocess

# 目标服务器A上的Agent控制电脑B上的程序
# 服务器A：运行Django、agent、MCP Client
# 电脑B：运行MCP server

# 1. 电脑B：MCP server

### 创建MCP与工具

（1）创建mcp服务（mcp_server.py中）
```python
# 创建服务
mcp = FastMCP("服务名")

# 注册工具
mcp.tool()(工具名)

# 启动服务
mcp.run_async(
    transport="streamable-http", # MCP Client 和 MCP Server 之间采用什么通信方式
    host="0.0.0.0",            # MCP Server监听哪个网络地址
    port=8000                  # 监听端口
)
```


（2）创建mcp工具（tools文件夹下单独实现功能）
```python
@mcp.tool()
def 工具名(参数):
    return 返回值
```

In [2]:
from fastmcp import FastMCP

# 创建MCP服务
mcp = FastMCP("calculator")


# 创建工具
@mcp.tool()
def add(a:int,b:int):
    return a+b


In [3]:
# 后台启动MCP
import asyncio

server_task = asyncio.create_task(
    mcp.run_async(
        transport="streamable-http", # MCP Client 和 MCP Server 之间采用什么通信方式
        host="0.0.0.0",            # MCP Server监听哪个网络地址
        port=8000                  # 监听端口
    )
)


print("电脑B MCP Server启动")

电脑B MCP Server启动




                    ╭──────────────────────────────────────────────────────────────────────────────╮                    
                    │                                                                              │                    
                    │                                                                              │                    
                    │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                    
                    │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                    
                    │                                                                              │                    
                    │                                                                              │                    
                    │                                                                              │                    
                    │         

[08/25/26 16:52:27] INFO     Starting MCP server 'calculator' with transport 'streamable-http' on  ]8;id=16584123;file://F:\Anaconda\envs\langchain\Lib\site-packages\fastmcp\server\mixins\transport.py\transport.py]8;;\:]8;id=16584124;file://F:\Anaconda\envs\langchain\Lib\site-packages\fastmcp\server\mixins\transport.py#361\361]8;;\
                             http://0.0.0.0:8000/mcp                                                               

INFO:     Started server process [12460]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:62666 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62669 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:62671 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62672 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62674 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62676 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62679 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:62680 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62682 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62684 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62686 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62726 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62729 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:62730 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62732 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62734 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62767 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:62770 -

In [ ]:
# 关闭MCP

server_task.cancel()

# 2. 服务器A：创建MCP Client

### 创建MCP客户端

（1）创建mcp客户端
```python
# 创建服务
client = Client("http://MCPserve电脑ip:端口/mcp")
```

（2）调用mcp工具（tools文件夹下单独实现功能）
```python
client.client.call_tool(
    "方法名",
    {
        "参数名":参数,
    }
```

In [4]:
from fastmcp import Client

client = Client(
    "http://127.0.0.1:8000/mcp"
)

In [5]:
# 服务器A发现电脑B工具

async with client:
    tools = await client.list_tools()
    for tool in tools:
        print(tool.name)

add


In [6]:
# 服务器A调用电脑B工具

async with client:
    result = await client.call_tool(
        "add",
        {
            "a":100,
            "b":200
        }
    )
    print(result)

CallToolResult(content=[TextContent(type='text', text='300', annotations=None, meta=None)], structured_content=None, meta=None, data=None, is_error=False)


# 3. 服务器A：将MCPtool封装为agenttool

In [17]:
from langchain_core.tools import StructuredTool

# 自动转换成agenttool

def convert_tool(mcp_tool):
    async def call(**kwargs):
        async with client:
            result = await client.call_tool(
                mcp_tool.name,
                kwargs
            )
            print(f"[Client]结果:{result.data}")
            return result.data

    return StructuredTool.from_function(
        coroutine=call,
        name=mcp_tool.name,
        description=mcp_tool.description or ""
    )


async def load_tools():
    async with client:
        tools = await client.list_tools()

    return [
        convert_tool(tool)
        for tool in tools
    ]

tools = await load_tools()
print(tools)

[StructuredTool(name='add', args_schema=<class 'langchain_core.utils.pydantic.add'>, coroutine=<function convert_tool.<locals>.call at 0x0000025E94309440>)]


In [10]:
# openai接口
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen3.5-plus",             # 模型名称
    openai_api_key="sk-bvOWJIfQ8Y8IrZhzzpQZ80zSKHjWIHCkPXEgQ0H4Li5UCpUC",         # API Key
    openai_api_base="http://ai.wenmodel.com/v1", # API 地址
    temperature=0.7,
    max_tokens=1024
)

response = llm.invoke("你是谁")
print(response.content)

你好！我是 Qwen3.5，是通义实验室研发的超大规模语言模型。作为通义千问系列的最新版本，我具备以下核心能力：

- **精准语言理解**：在各类场景中提供自然流畅的交互体验
- **多语言支持**：覆盖全球 100+ 主流语言，包括中文、英语、小语种等
- **代码全栈能力**：从代码生成、调试到可运行前端页面开发
- **逻辑推理强化**：在数学、科学领域提供专业级解答
- **视觉深度解析**：能分析图表、公式及科学图示的因果关系
- **长窗口掌控**：原生支持 256K 上下文，精准处理超长文档
- **智能体自主规划**：可独立完成多步骤任务（如搜索、代码执行）
- **专业领域优化**：在医疗、法律等垂直领域提供高质量回答

我经过严格对齐训练，能严格遵循指令，提供安全可靠的回答。无论是写故事、写公文、写邮件、写剧本，还是逻辑推理、编程、表达观点、玩游戏，我都能尽力协助。

有什么具体任务需要我帮你完成吗？ 😊


In [13]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=llm,
    tools=tools
)

<frozen abc>:117: RuntimeWarning: coroutine 'load_tools' was never awaited


In [18]:
result = await agent.ainvoke(
    {
        "messages":[
            {
                "role":"user",
                "content":
                "请计算100+200"
            }
        ]
    }
)


print(
    result["messages"][-1].content
)

100 + 200 = 300
